In [2]:
import polars as pl
import os

In [3]:
def sample_data(dataframe: pl.DataFrame,
                n_samples: int,
                threshold: int = 0,
                seed: int = 42) -> pl.DataFrame:
    """
    Samples n_samples rows from the given DataFrame.

    Args:
        dataframe (pl.DataFrame):
            The input DataFrame to sample from.
        n_samples (int):
            The number of samples to draw.
        threshold (int):
            The minimum value for 'raw_sequence_length' to consider a row 
            for sampling.
        seed (int):
            The random seed for reproducibility. To re-sample differently,
            change this value.

    Returns:
        pl.DataFrame: A DataFrame containing the sampled rows.
    """
    sampled_df = dataframe.filter(pl.col("raw_sequence_length") > threshold)
    sampled_df = sampled_df.sample(n=n_samples,
                                   seed=seed)
    return sampled_df

In [10]:
base_path = "output/" # Adjust the base path as needed 
save_path = "sampled_output/" # Adjust the save path as needed

if not os.path.exists(save_path):
    os.makedirs(save_path)

In [26]:
# change accordingly
threshold = 0
n_samples = 205
seed = 42 # Change this value to re-sample differently

# Iterate over languages and subsets to sample and save data
for language in ["de"]: #, "fr", "it"]:
    for subset in ["llama4scout_"]:
    #["corpus_", "gpt-4.1-mini_", "llama3.1-8B-instruct_", "llama4scout_", "occiglot-7b-eu5-instruct_"]:
        df = pl.read_csv(f"{base_path}{subset}{language}_features.csv")
        sampled_df = sample_data(df,
                                 n_samples=n_samples,
                                 threshold=threshold,
                                 seed=seed)
        
        # Select only relevant columns before saving
        if subset == "corpus_":
            sampled_df = sampled_df.select(
                ["argument_id", "question", "argument"])
        else:
            sampled_df = sampled_df.select(
                ["output_id", "id", "output"]
            )
        sampled_df.write_csv(
            f"{save_path}{subset}{language}_sample.csv"
        )

In [ ]:
# DE
# Replace prompt id with political issue question
df_prompts = pl.read_csv("prompts_de.csv")

# llama4scout_de
path = "sampled_output/llama4scout_de_sample.csv"
df_sample = pl.read_csv(path)
df_sample = df_sample.join(df_prompts, on="id", how="left")
df_sample = df_sample.select(["output_id", "question", "output"])

df_sample.write_csv(path)


In [30]:
# gpt-4.1-mini_de
path = "sampled_output/gpt-4.1-mini_de_sample.csv"
df_sample = pl.read_csv(path)
df_sample = df_sample.rename({"prompt_id": "id"})
df_sample = df_sample.join(df_prompts, on="id", how="left")
df_sample = df_sample.select(["output_id", "question", "output"])

df_sample.write_csv(path)